## 인물 보존형 AI 가상 배경 스튜디오

In [ ]:
!pip install -q transformers dataset
!pip install -q sentencepiece
!pip install -q kobert-transformers
!pip install -q controlnet-aux

In [ ]:
import torch
from PIL import Image, ImageOps
from io import BytesIO
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, UniPCMultistepScheduler
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TranslationPipeline # class
from controlnet_aux import CannyDetector # 허깅페이스 보조 라이브러리(이미지의 외각선 추출)
import numpy as np
import cv2
import matplotlib.pyplot as plt


### 외곽선 추출 모델 및 이미지 생성
- 외각선 추출 도구 CannyDetector

In [ ]:
processor = CannyDetector()
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained( "runwayml/stable-diffusion-v1-5", controlnet=controlnet,
    torch_dtype=torch.float16).to("cuda")

##### 속도 최적화

In [ ]:
# 속도 최적화를 위한 스케줄러 설정
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

### 원본이미지 전처리

##### 배경제거 모델 활용

In [ ]:
rmbg_pipe = pipeline(
    "image-segmentation",
    model="briaai/RMBG-1.4",
    trust_remote_code=True
)

In [ ]:
init_image = Image.open('test_man.jpg').convert('RGB').resize((512, 768)) # 홈페이지 올리는 사이즈로 수정하기
canny_image = processor(init_image, low_threshold=150, high_threshold=250)

### 프롬프트 번역 모델(한글 -> 영어)

In [ ]:
model_name = 'Helsinki-NLP/opus-mt-ko-en'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
translator = TranslationPipeline(model=model, tokenizer = tokenizer)

# 3. 사용자 한국어 입력 (원하는 것 / 빼고 싶은 것)
ko_prompt = "눈부시게 밝은 정오의 햇살, 투명한 터콰이즈 빛 바다 위에 떠 있는 모습, 수면에 반사되는 강렬한 빛, 하얀 모래사장"
ko_negative = "어두운 방, 검은 배경, 스튜디오 조명, 그림자, 밤, 저녁, 초록색 식물, 정원, 나무, 저화질, 지저분한 배경"

# 4. 번역 실행
# 번역 모델은 리스트 형태로 한꺼번에 넣으면 효율적
results = translator([ko_prompt, ko_negative])
en_prompt = results[0]['translation_text']
en_negative = results[1]['translation_text']

# 생성 모델이 더 잘 알아듣도록 품질 키워드를 강제로 추가 (퀄리티 보장용)
final_prompt = f"{en_prompt}, high-key lighting, extremely bright, sunny day, crystal clear water, highly detailed, masterpiece, 8k resolution, sharp focus, (detailed black hair:1.4), male hairstyle, (refined facial features:1.2), masterpiece, 8k, sharp focus"
final_negative = f"{en_negative}, pitch black, dark background, shadow, moody, low-light, grainy, bad anatomy, watermark hairless, head shine, (distorted head:1.3), watermark"
print(f"최종 프롬프트: {final_prompt}")
print(f"최종 네거티브: {final_negative}")

#### 인물 누끼 및 마스크 보정

In [ ]:

# 1. 누끼 따기 및 마스크 반전
mask = rmbg_pipe(init_image, return_mask=True)
mask = ImageOps.invert(mask) # 인물은 검정(0), 배경은 흰색(255)
mask_np = np.array(mask)

# [핵심] 얼굴과 머리 윗부분(이미지 상단 약 30%)은 강제로 보존(검은색) 처리
# 인물의 위치에 따라 height 비율을 0.3~0.4 사이로 조절
h, w = mask_np.shape
mask_np[0:int(h*0.35), :] = 0

# 2. 마스크 경계 부드럽게 (합성 티 제거)
mask_blurred = cv2.GaussianBlur(mask_np, (15, 15), 0)
mask = Image.fromarray(mask_blurred)

##### 부정 프롬프트 :  AI가 흔히 저지르는 실수를 미리 차단하는 역할/ 특히 Img2Img나 ControlNet 사용 시 인물이 망가지는 것을 방지하기 위해 필수적
- 인체 왜곡 방지: bad anatomy, extra fingers, deformed limbs (손가락이나 팔다리 꼬임 방지)
- 화질 저하 방지: lowres, text, error, cropped, worst quality, low quality, jpeg artifacts
- 배경 혼선 방지: dark forest, garden, green leaves (원본 이미지에 있던 요소를 지우고 싶을 때 명시)


##### 긍정 프롬프트 (Prompt) : 생성형 AI는 문장의 앞부분에 있는 단어에 더 큰 비중을 둡니다. 따라서 [핵심 피사체] -> [배경] -> [조명/분위기] -> [화질 키워드] 순서로 배치하는 것이 정석
- 핵심 피사체 유지: 지금처럼 인물을 고정하고 싶다면, 원본 인물의 특징을 짧게라도 적어주는 게 좋습니다. (예: a woman in a floral jumpsuit)
- 배경 묘사 구체화: 단순한 space보다는 색감이나 요소를 더해보세요.
- 조명(Lighting): 배경과 인물을 자연스럽게 섞어주는 핵심 요소

### 배경생성

In [ ]:

# controlnet_conditioning_scale: 1.0~1.2 (인물의 형태를 얼마나 엄격하게 지킬지)
output = pipe(
    prompt = final_prompt,
    negative_prompt = final_negative,
    image = init_image,
    mask_image = mask,
    control_image = canny_image,
    num_inference_steps = 80,

    # [수정] 강도를 살짝 낮춰서 배경색이 인물 경계에 스며들게
    controlnet_conditioning_scale = 0.85,

    # [핵심] 가이드를 0.5에서 끊어서, 나머지 50% 과정 동안
    # AI가 인물 외곽의 잔상을 바다색으로 자연스럽게 덮게 만듦
    control_guidance_end = 0.5,

    # 프롬프트 가이던스를 살짝 낮춰서 색 대비가 너무 튀지 않게 조절
    guidance_scale = 9.0,

    strength = 1.0
).images[0]

# 5. 결과 확인
output.save("huggingface_space_result2.png")
output.show()

- strength (중요): 원본 이미지를 얼마나 변형할지 결정합니다. 배경만 자연스럽게 바꾸려면 0.5 ~ 0.6 정도가 적당, 값이 너무 높으면 원본 피사체의 형태까지 무너질 수 있음
- guidance_scale: 프롬프트(텍스트)를 얼마나 엄격하게 따를지 결정합니다. 보통 7.5 ~ 10 사이를 사용함
- negative_prompt: 결과물에서 제외하고 싶은 요소(저화질, 왜곡 등)를 적어주면 품질이 올라감

In [ ]:

def show_comparison(init, result):
    plt.figure(figsize=(15, 7)) # 가로 15, 세로 7 크기의 도화지

    # 왼쪽 - 원본 이미지
    plt.subplot(1, 2, 1)
    plt.imshow(init)
    plt.title("Original Image (Before)", fontsize=15)
    plt.axis('off') # 격자 제거

    # 오른쪽 -  생성된 결과물
    plt.subplot(1, 2, 2)
    plt.imshow(result)
    plt.title("Generated Studio (After)", fontsize=15)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# 실행
show_comparison(init_image, output)